<a href="https://colab.research.google.com/github/pinkypnair28/nyc-311-data-governance-audit/blob/main/data_governance_simulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install --upgrade google-cloud-bigquery -q

from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
import pandas as pd

PROJECT_ID = "data-governance-simulation"  # the exact project ID from Step 1 (check the project picker dropdown)
TABLE = "bigquery-public-data.new_york.311_service_requests"

client = bigquery.Client(project=PROJECT_ID)
print("Connected ✅")

Connected ✅


In [ ]:
def run_query(sql):
    return client.query(sql).to_dataframe()

def check_completeness():
    sql = f"""
    SELECT
      COUNTIF(incident_zip IS NULL) / COUNT(*) AS pct_missing_zip,
      COUNTIF(borough IS NULL OR borough = 'Unspecified') / COUNT(*) AS pct_missing_borough,
      COUNTIF(closed_date IS NULL) / COUNT(*) AS pct_missing_closed_date,
      COUNTIF(descriptor IS NULL) / COUNT(*) AS pct_missing_descriptor
    FROM `{TABLE}`
    """
    return run_query(sql)

def check_uniqueness():
    sql = f"""
    SELECT COUNT(*) AS duplicate_key_count
    FROM (
      SELECT unique_key FROM `{TABLE}` GROUP BY unique_key HAVING COUNT(*) > 1
    )
    """
    return run_query(sql)

def check_temporal_validity():
    sql = f"""
    SELECT COUNT(*) AS invalid_date_records
    FROM `{TABLE}`
    WHERE closed_date < created_date
    """
    return run_query(sql)

def check_conformity():
    sql = f"""
    SELECT
      LOWER(TRIM(REGEXP_REPLACE(complaint_type, r'[^a-zA-Z0-9 ]', ''))) AS normalized_label,
      COUNT(DISTINCT complaint_type) AS raw_variant_count,
      ARRAY_AGG(DISTINCT complaint_type LIMIT 5) AS example_variants
    FROM `{TABLE}`
    GROUP BY normalized_label
    HAVING raw_variant_count > 1
    ORDER BY raw_variant_count DESC
    """
    return run_query(sql)

In [ ]:
completeness = check_completeness()
uniqueness = check_uniqueness()
temporal = check_temporal_validity()
conformity = check_conformity()

print("=== Data Quality Report: NYC 311 Service Requests ===\n")
print("1. Completeness"); display(completeness)
print(f"\n2. Uniqueness — duplicate keys: {uniqueness['duplicate_key_count'][0]}")
print(f"\n3. Temporal validity — invalid records: {temporal['invalid_date_records'][0]:,}")
print(f"\n4. Conformity — duplicate clusters: {len(conformity)}"); display(conformity)

=== Data Quality Report: NYC 311 Service Requests ===

1. Completeness


,pct_missing_zip,pct_missing_borough,pct_missing_closed_date,pct_missing_descriptor
0,0.048106,0.041948,0.027923,0.016085



2. Uniqueness — duplicate keys: 0

3. Temporal validity — invalid records: 402,189

4. Conformity — duplicate clusters: 23


,normalized_label,raw_variant_count,example_variants
0,,8,"[../../../../../../../../../..., .../...//.../..."
1,misc comments,4,"[Misc. Comments../../../../...., .../Misc. Com..."
2,webinfwebxml,4,"[../../../../WEB-INF/web.xml, ../WEB-INF/web.x..."
3,misc commentsdeclare q,3,"[Misc. Comments);declare @q ..., Misc. Comment..."
4,webinfwebxmlx,3,"[../WEB-INF/web.xml;x=, ../../../WEB-INF/web.x..."
5,outside building,2,"[Outside Building, OUTSIDE BUILDING]"
6,misc commentssleep20,2,"[Misc. Comments'.sleep(20).', Misc. Comments{$..."
7,lead,2,"[Lead, LEAD]"
8,elevator,2,"[ELEVATOR, Elevator]"
9,mold,2,"[Mold, MOLD]"


In [ ]:
def score_report(completeness, uniqueness, temporal, conformity):
    scorecard = []
    for col in completeness.columns:
        pct_complete = 1 - completeness[col][0]
        scorecard.append({
            "rule": f"Completeness: {col.replace('pct_missing_', '')}",
            "result": f"{pct_complete:.1%}",
            "status": "PASS" if pct_complete >= 0.98 else "FAIL"
        })
    dup_count = uniqueness['duplicate_key_count'][0]
    scorecard.append({"rule": "Uniqueness: unique_key", "result": f"{dup_count} duplicates",
                       "status": "PASS" if dup_count == 0 else "FAIL"})
    invalid_dates = temporal['invalid_date_records'][0]
    scorecard.append({"rule": "Validity: closed_date >= created_date", "result": f"{invalid_dates:,} violations",
                       "status": "PASS" if invalid_dates == 0 else "FAIL"})
    dup_clusters = len(conformity)
    scorecard.append({"rule": "Conformity: complaint_type taxonomy", "result": f"{dup_clusters} duplicate clusters",
                       "status": "PASS" if dup_clusters == 0 else "FAIL"})
    return pd.DataFrame(scorecard)

score_report(completeness, uniqueness, temporal, conformity)

,rule,result,status
0,Completeness: zip,95.2%,FAIL
1,Completeness: borough,95.8%,FAIL
2,Completeness: closed_date,97.2%,FAIL
3,Completeness: descriptor,98.4%,PASS
4,Uniqueness: unique_key,0 duplicates,PASS
5,Validity: closed_date >= created_date,"402,189 violations",FAIL
6,Conformity: complaint_type taxonomy,23 duplicate clusters,FAIL
